# 01 — Data Quality, Schema Governance & Financial Reconciliation Audit

## Objective
Audit all five relational tables in the MedTourEasy Patient Intelligence dataset across CSV and SQLite sources:
1. Verify row counts, column data types, missing values, and unique cardinalities.
2. Audit primary-key uniqueness and foreign-key referential integrity.
3. Reconcile CSV records against `mte_patient_intelligence.db`.
4. Reconcile completed treatments, actual revenue, service cost, and gross margin between `patient_journey` and `treatment_outcomes`.
5. Document structural missingness and funnel stage tracking anomalies (`229` completed treatments with `quote_shared=0`).

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'

from src.data_quality import (
    load_csvs,
    load_sqlite,
    profile_all_tables,
    audit_primary_keys,
    audit_foreign_keys,
    reconcile_csv_sqlite,
    reconcile_financials,
    detect_data_anomalies,
)

In [2]:
csv_tables = load_csvs(DATA)
db_tables = load_sqlite(DATA / 'mte_patient_intelligence.db')
for name, df in csv_tables.items():
    print(f'{name:22s} | CSV shape: {str(df.shape):12s} | SQLite shape: {db_tables[name].shape}')

patient_journey        | CSV shape: (1500, 26)   | SQLite shape: (1500, 26)
provider_master        | CSV shape: (30, 14)     | SQLite shape: (30, 14)
communication_logs     | CSV shape: (6025, 10)   | SQLite shape: (6025, 10)
treatment_outcomes     | CSV shape: (554, 10)    | SQLite shape: (554, 10)
country_reference      | CSV shape: (12, 7)      | SQLite shape: (12, 7)


## 1. Primary Key & Foreign Key Integrity Audits

In [3]:
pk_audit = audit_primary_keys(csv_tables)
display(pk_audit)
fk_audit = audit_foreign_keys(csv_tables)
display(fk_audit)

,table,primary_key,null_keys,duplicate_keys,unique_keys,row_count,unique_ok
0,patient_journey,patient_id,0,0,1500,1500,True
1,provider_master,provider_id,0,0,30,30,True
2,communication_logs,communication_id,0,0,6025,6025,True
3,treatment_outcomes,patient_id,0,0,554,554,True
4,country_reference,country,0,0,12,12,True


,relationship,orphan_count
0,patient_journey.provider_id -> provider_master...,0
1,patient_journey.country -> country_reference.c...,0
2,communication_logs.patient_id -> patient_journ...,0
3,communication_logs.provider_id -> provider_mas...,0
4,treatment_outcomes.patient_id -> patient_journ...,0
5,treatment_outcomes.provider_id -> provider_mas...,0


## 2. Column-Level Data Quality Profile & Structural Missingness

In [4]:
dq_profile = profile_all_tables(csv_tables)
display(dq_profile[dq_profile['null_count'] > 0])
display(dq_profile.head(26))

,table,column,dtype,rows,null_count,null_pct,unique_count
20,patient_journey,drop_off_stage,object,1500,1150,76.666667,6
21,patient_journey,reason_for_drop_off,object,1500,1150,76.666667,10
25,patient_journey,satisfaction_score,float64,1500,946,63.066667,56
48,communication_logs,response_delay_hours,float64,6025,1413,23.452282,460


,table,column,dtype,rows,null_count,null_pct,unique_count
0,patient_journey,patient_id,object,1500,0,0.000000,1500
1,patient_journey,inquiry_date,object,1500,0,0.000000,1350
2,patient_journey,country,object,1500,0,0.000000,12
3,patient_journey,age_group,object,1500,0,0.000000,5
4,patient_journey,gender,object,1500,0,0.000000,3
5,patient_journey,treatment_category,object,1500,0,0.000000,9
6,patient_journey,provider_id,object,1500,0,0.000000,30
7,patient_journey,lead_source,object,1500,0,0.000000,6
8,patient_journey,coordinator_id,object,1500,0,0.000000,15
9,patient_journey,budget_band,object,1500,0,0.000000,4


## 3. Financial Reconciliation (`patient_journey` vs. `treatment_outcomes`)

In [5]:
fin_recon = reconcile_financials(csv_tables['patient_journey'], csv_tables['treatment_outcomes'])
display(fin_recon)

,metric,patient_journey_value,treatment_outcomes_value,reconciled
0,Completed patient count,554,554,1
1,Completed revenue total,248325245,248325245,True
2,Completed service cost total,136810889,136810889,True
3,Gross margin formula,111514356,111514356,True
4,Completed patients with nonzero revenue,554,554,True
5,Incomplete patients with nonzero revenue,0,0,True


## 4. Automated Governance & Anomaly Log

In [6]:
anomalies = detect_data_anomalies(csv_tables)
display(anomalies)

,area,finding,severity,recommended_control
0,SQLite schema,All 5 tables have columns declared as TEXT; nu...,Governance risk,Cast numeric/date fields in production queries...
1,SQLite constraints,PRAGMA foreign_keys=0; no declared indexes/for...,Governance risk,Add PK/FK constraints and indexes in a product...
2,patient_journey.satisfaction_score,946 missing values (63.1%). Missingness occurs...,Structural missingness,Treat as post-treatment outcome; do not impute...
3,communication_logs.response_delay_hours,1413 missing values (23.5%). Every missing val...,Structural missingness,Retain as null when no response was sent; do n...
4,Funnel tracking,229 completed patients have quote_shared=0.,Stage integrity anomaly,Audit quote capture; report raw flags and a st...
5,Drop-off coding,350 rows are explicitly coded as drop-offs; co...,Definition mismatch,Keep both definitions: coded drop-off and stag...
6,Financial reconciliation,patient_journey revenue/cost exactly reconcile...,No anomaly,Use treatment_outcomes as outcome-level audit ...
7,Treatment timing,0 treatment dates precede inquiry dates.,No anomaly,No pre-inquiry treatment dates were detected.
